# SLAM interface

This notebook wraps the same SLAM client, subscribers, point-cloud decoding, and Plotly map helpers used by `../scripts/slam_web_app.py` into a Jupyter control panel.


In [1]:
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
MODULES_DIR = NOTEBOOK_DIR.parent
ROOT_DIR = MODULES_DIR.parent
for path in (str(MODULES_DIR), str(ROOT_DIR), str(MODULES_DIR / "scripts")):
    if path not in sys.path:
        sys.path.insert(0, path)

IFACE = os.environ.get("G1_IFACE", "eth0")
DOMAIN_ID = int(os.environ.get("G1_DOMAIN_ID", "0"))
print(f"Configured for iface={IFACE!r}, domain_id={DOMAIN_ID}.")


Configured for iface='eth0', domain_id=0.


Import the reusable SLAM web-app classes rather than duplicating the point-cloud and task-navigation code.


In [2]:
import json
import os
import time

import ipywidgets as widgets
from IPython.display import display

from slam_web_app import DEFAULT_TOPICS, LAYER_STYLE, SlamWebState, make_figure


Create the SLAM state object. Change `MAP_PATH` if your exercise uses a different map location.


In [3]:
MAP_PATH = os.environ.get("G1_SLAM_MAP_PATH", str(MODULES_DIR / "maps" / "academy_map"))
slam = SlamWebState(IFACE, DOMAIN_ID, dict(DEFAULT_TOPICS), MAP_PATH)
print(f"SLAM state ready. map_path={MAP_PATH}")


SLAM state ready. map_path=/home/unitree/EF/ef_ws/g1/modules/maps/academy_map


These helpers format status and rebuild the Plotly map with the selected layers.


In [4]:
def status_json():
    return json.dumps(slam.status(), indent=2, default=str, sort_keys=True)


def redraw():
    fig = make_figure(slam, list(layers.value), int(max_points.value), view_mode.value)
    fig.update_layout(height=650)
    return fig


Run the panel. Mapping, relocation, task queueing, and selected-pose navigation call the same `SlamOperateClient` methods as the Dash app.


In [5]:
slam_type = widgets.Dropdown(options=["mapping", "navigation"], value="mapping", description="SLAM")
map_path = widgets.Text(value=MAP_PATH, description="Map", layout=widgets.Layout(width="520px"))
layers = widgets.SelectMultiple(
    options=list(LAYER_STYLE.keys()),
    value=("slam_mapping", "slam_relocation", "occupancy"),
    description="Layers",
    layout=widgets.Layout(height="170px"),
)
max_points = widgets.IntSlider(value=18000, min=2000, max=60000, step=2000, description="Points")
view_mode = widgets.ToggleButtons(options=["world", "sensor"], value="world", description="View")
start_mapping = widgets.Button(description="Start Mapping", button_style="success")
save_map = widgets.Button(description="Save Map", button_style="primary")
relocate = widgets.Button(description="Relocate", button_style="info")
stop_slam = widgets.Button(description="Stop SLAM", button_style="warning")
add_current = widgets.Button(description="Add Current Pose")
go_selected = widgets.Button(description="Go Selected", button_style="danger")
execute_tasks = widgets.Button(description="Execute Tasks", button_style="danger")
clear_tasks = widgets.Button(description="Clear Tasks")
refresh = widgets.Button(description="Refresh")
status_box = widgets.Textarea(layout=widgets.Layout(width="100%", height="260px"), disabled=True)
out = widgets.Output()


def refresh_all(message=None):
    if message:
        slam.last_action = {"label": "notebook", "ok": True, "raw": message, "stamp": time.time()}
    status_box.value = status_json()
    with out:
        out.clear_output(wait=True)
        display(redraw())


def call(action):
    def _handler(_):
        try:
            if action == "start":
                slam.start_mapping(slam_type.value)
            elif action == "save":
                slam.save_map(map_path.value)
            elif action == "relocate":
                slam.relocate(map_path.value)
            elif action == "stop":
                slam.stop_slam()
            elif action == "add_current":
                slam.add_current_pose()
            elif action == "go_selected":
                slam.go_to_selected_pose()
            elif action == "execute":
                slam.execute_tasks()
            elif action == "clear":
                slam.clear_tasks()
        except Exception as exc:
            slam.last_action = {"label": action, "ok": False, "raw": str(exc), "stamp": time.time()}
        refresh_all()
    return _handler

for button, action in [
    (start_mapping, "start"), (save_map, "save"), (relocate, "relocate"), (stop_slam, "stop"),
    (add_current, "add_current"), (go_selected, "go_selected"), (execute_tasks, "execute"), (clear_tasks, "clear"),
]:
    button.on_click(call(action))
refresh.on_click(lambda _: refresh_all())
refresh_all("Panel ready.")
display(widgets.VBox([
    widgets.HBox([slam_type, map_path]),
    widgets.HBox([start_mapping, save_map, relocate, stop_slam, refresh]),
    widgets.HBox([add_current, go_selected, execute_tasks, clear_tasks]),
    widgets.HBox([layers, widgets.VBox([max_points, view_mode])]),
    status_box,
    out,
]))
